# 05 — Model Evaluation
## Credit Risk: Loss Given Default (LGD) Prediction

**Objectives:**
- Calibration decile plot: predicted vs. actual LGD across the full range
- SHAP explainability: global feature importance and individual loan explanations
- Macro sensitivity: verify LGD rises with unemployment and falls with HPI
- Residual analysis: identify systematic errors by segment
- Final success criteria assessment

## Setup

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

sys.path.insert(0, str(Path.cwd().parent))

from src.utils.config import load_config
from src.models.evaluate import (
    load_model, calibration_plot, shap_analysis,
    macro_scenario_analysis, compute_metrics
)
from src.models.train import prepare_data

config = load_config()
device = torch.device('cpu')

%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 6)
print('Setup complete')

## 1. Load Model and Test Data

In [ ]:
import pickle
from sklearn.model_selection import train_test_split

model_path = Path(config.api.model_path)
scaler_path = Path('models') / 'scaler.pt'

if not model_path.exists():
    print('Model not found. Run notebook 04 or: python src/models/train.py')
else:
    model = load_model(model_path, device)
    print('Model loaded successfully')
    
    if scaler_path.exists():
        with open(scaler_path, 'rb') as f:
            scaler = pickle.load(f)
        print('Scaler loaded')
    
    X_train, X_val, X_test, y_train, y_val, y_test, scaler_train = prepare_data(config)
    
    X_test_scaled = scaler.transform(X_test) if scaler_path.exists() else scaler_train.transform(X_test)
    
    x_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
    model.eval()
    with torch.no_grad():
        y_pred = model(x_tensor).numpy()
    
    test_metrics = compute_metrics(y_test, y_pred)
    print(f'\nTest MAE:  {test_metrics["mae"]:.4f}')
    print(f'Test RMSE: {test_metrics["rmse"]:.4f}')
    print(f'Test R²:   {test_metrics["r2"]:.4f}')

## 2. Calibration Decile Plot

A well-calibrated model tracks the diagonal. Regulatory stress testing requires calibration to hold in the tails.

In [ ]:
# TODO: Generate and display calibration plot
fig = calibration_plot(y_test, y_pred, n_bins=10, save_path='../reports/calibration_plot.png')
plt.show()

## 3. SHAP Feature Importance

In [ ]:
# TODO: SHAP analysis
# Expected top features: ltv_at_default, hpi_change, occupancy_status, orig_ltv

feature_names = [c for c in config.features.categorical + config.features.numeric]

try:
    import shap
    bg_idx = np.random.choice(len(X_train), size=min(200, len(X_train)), replace=False)
    scaler_active = scaler if scaler_path.exists() else scaler_train
    X_bg = scaler_active.transform(X_train[bg_idx])
    
    explain_idx = np.random.choice(len(X_test_scaled), size=min(300, len(X_test_scaled)), replace=False)
    X_explain = X_test_scaled[explain_idx]
    
    shap_values = shap_analysis(
        model, X_bg, X_explain, feature_names,
        save_dir='../reports'
    )
    
    if shap_values is not None:
        print('SHAP analysis complete. Plots saved to reports/')
except ImportError:
    print('SHAP not installed: pip install shap')

## 4. Macro Sensitivity Analysis

**Key regulatory requirement:** Predicted LGD must rise monotonically as unemployment increases and HPI falls.

In [ ]:
# TODO: Macro scenario sensitivity
scaler_active = scaler if scaler_path.exists() else scaler_train

scenario_results = macro_scenario_analysis(
    model, X_test, feature_names, scaler_active,
    save_path='../reports/macro_scenarios.png'
)

plt.show()

if not scenario_results.empty:
    print('Macro scenario results:')
    print(scenario_results.to_string(index=False))
    
    # Verify monotonic increase
    lgd_values = scenario_results['mean_lgd'].values
    is_monotonic = all(lgd_values[i] <= lgd_values[i+1] for i in range(len(lgd_values)-1))
    print(f'\nMonotonic increase (stress → more LGD): {is_monotonic}')
    if not is_monotonic:
        print('WARNING: LGD does not increase monotonically with stress — investigate model')

## 5. Residual Analysis

In [ ]:
# TODO: Residuals by predicted LGD range
residuals = y_pred - y_test

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Residuals vs predicted
axes[0].scatter(y_pred, residuals, alpha=0.1, s=5, color='#2563EB')
axes[0].axhline(0, color='red', linestyle='--', alpha=0.7)
axes[0].set_xlabel('Predicted LGD')
axes[0].set_ylabel('Residual (Predicted - Actual)')
axes[0].set_title('Residuals vs Predicted LGD')
axes[0].grid(True, alpha=0.3)

# Residual distribution
axes[1].hist(residuals, bins=50, color='#2563EB', alpha=0.8, edgecolor='white')
axes[1].axvline(0, color='red', linestyle='--')
axes[1].set_xlabel('Residual')
axes[1].set_ylabel('Count')
axes[1].set_title(f'Residual Distribution (mean={residuals.mean():.4f}, std={residuals.std():.4f})')

plt.tight_layout()
plt.show()

# Large error analysis
large_errors = np.abs(residuals) > 0.3
print(f'Predictions with |error| > 0.3: {large_errors.sum():,} ({large_errors.mean()*100:.1f}%)')

## 6. Success Criteria Assessment

In [ ]:
# TODO: Fill in after running all evaluations
# Replace placeholder values with actuals

benchmark_mae = None   # From notebook 03
benchmark_rmse = None  # From notebook 03
model_mae = test_metrics['mae']
model_rmse = test_metrics['rmse']
model_r2 = test_metrics['r2']

print('=== SUCCESS CRITERIA ASSESSMENT ===')
print()

criteria = [
    ('MAE ≥ 15% improvement vs benchmark', 
     f"{(benchmark_mae - model_mae) / benchmark_mae * 100:.1f}%" if benchmark_mae else 'TBD',
     '≥ 15%'),
    ('RMSE ≥ 10% improvement vs benchmark',
     f"{(benchmark_rmse - model_rmse) / benchmark_rmse * 100:.1f}%" if benchmark_rmse else 'TBD',
     '≥ 10%'),
    ('R² > 0.30 on test set',
     f'{model_r2:.4f}',
     '> 0.30'),
    ('Calibration holds in tails', 'VISUAL CHECK', 'See plot above'),
    ('Macro sensitivity: stress → higher LGD', 'TBD from scenario analysis', 'Monotonic increase'),
    ('API latency < 200ms', 'TBD after API deployment', '< 200ms'),
]

for criterion, actual, target in criteria:
    print(f'{criterion}')
    print(f'  Target: {target} | Actual: {actual}')
    print()

## Key Findings

*To be completed after full evaluation:*

1. **Test MAE / RMSE / R²:** [values]
2. **Improvement vs segment-average benchmark:** [MAE %, RMSE %]
3. **Calibration assessment:** [Does it hold in the tails?]
4. **SHAP top features:** [Top 5 features by mean |SHAP|]
5. **SHAP directional coherence:** [Do feature effects match domain expectations?]
6. **Macro sensitivity result:** [Does LGD rise monotonically under stress?]
7. **Overall verdict:** [Does LGDNet meet all success criteria?]